# Generate synthetic tables for a Genie demo

Reads the statistics zip produced by `01_profile_tables` in the customer workspace and writes fake tables
with the same names, columns, types and comments into a catalog and schema of your choice. The tables
join like the originals (declared primary and foreign keys), and columns follow the original
distributions. Nothing here touches customer data: the zip contains statistics only.

**How to use it**
1. Upload `synthgen_profiles.zip` to a volume or workspace folder.
2. Edit **cell 2 (Configuration)**: where the zip is, and the target catalog and schema.
3. Click **Run all**. Cell 4 prints the plan (tables, sizes, links) before anything is written.
4. Check the summary and sample rows at the end, then point a Genie space at the new schema.

## 1. Install the one dependency (pydantic)

In [ ]:
%pip install -q "pydantic>=2.7"

In [ ]:
dbutils.library.restartPython()  # loads the freshly installed package

In [ ]:
# ============================== 2. CONFIGURATION: edit, then run =================================

# The zip received from the customer (or the folder with the unzipped .json files).
PROFILES = "/Volumes/main/genie_demo/files/synthgen_profiles.zip"

# Where to create the fake tables. The schema is created if it does not exist.
TARGET_CATALOG = "main"
TARGET_SCHEMA = "genie_demo"

# Size: each table gets SCALE x its real row count, at least MIN_ROWS (or its real size if smaller).
# Child tables (orders, order lines...) follow from their parents, keeping the real proportions.
SCALE = 0.01
MIN_ROWS = 100
MAX_ROWS = 1_000_000  # hard cap per table (rows are generated on the driver)

SEED = 42  # same seed + same zip = same data

# Declare primary and foreign keys on the new tables so Genie knows how they join (Unity Catalog).
DECLARE_KEYS = True

# Safety: tables this notebook did not create are never replaced unless you set this to True.
REPLACE_EXISTING = False

TABLE_FORMAT = "delta"

In [ ]:
# ============================== 3. ENGINE: run, do not edit ======================================
# The synthgen code, embedded so nothing has to be uploaded. It is plain Python: feel free to read it.
# It only defines functions; nothing is read or written in this cell.

import sys
import types


def _load(name, source, package=False):
    module = types.ModuleType(name)
    module.__file__ = "<synthgen notebook: " + name + ">"
    if package:
        module.__path__ = []
    sys.modules[name] = module
    exec(compile(source, module.__file__, "exec"), module.__dict__)


_load("synthgen", r'''
"""synthgen: profile tables into privacy-filtered statistics, regenerate synthetic look-alikes."""

__version__ = "0.1.0"
''', package=True)

_load("synthgen.config", r'''
"""Dataset configuration.

Every threshold the toolkit uses lives here. Defaults are declared once on these models and
`config/example_dataset.yaml` spells all of them out, so a dataset owner can see and tune each one.
Code never hard-codes a threshold; it reads it from a `SynthgenConfig` instance.
"""

from __future__ import annotations

from pathlib import Path
from typing import Literal

from pydantic import BaseModel, ConfigDict, Field, field_validator, model_validator

# Hard floor for k. The plan allows configuring k but never below 5.
MIN_K = 5


class _Section(BaseModel):
    model_config = ConfigDict(extra="forbid")


class SourceConfig(_Section):
    tables: list[str] = Field(default_factory=list, description="Explicit catalog.schema.table names.")
    table_patterns: list[str] = Field(
        default_factory=list, description="Glob patterns on catalog.schema.table, e.g. 'main.sales.*'."
    )
    exclude_patterns: list[str] = Field(default_factory=list)
    include_views: bool = False

    @model_validator(mode="after")
    def _non_empty(self) -> SourceConfig:
        if not self.tables and not self.table_patterns:
            raise ValueError("source needs at least one of 'tables' or 'table_patterns'")
        for name in self.tables:
            if name.count(".") != 2:
                raise ValueError(f"table '{name}' must be fully qualified as catalog.schema.table")
        return self


class OutputConfig(_Section):
    root: str = Field(description="Folder the profiler may write to. Profiles land in <root>/review/.")


DEFAULT_PII_NAME_PATTERNS: dict[str, list[str]] = {
    # semantic type -> regexes matched against the lower-cased column name.
    "person_name": [
        r"(^|_)(first|last|full|given|family|middle|maiden|customer|contact|person|user|owner|holder)_?name($|_)",
        r"^name$",
        r"(^|_)(surname|forename|firstname|lastname)($|_)",
    ],
    "email": [r"e_?mail"],
    "phone": [r"phone", r"mobile", r"(^|_)(tel|telephone|fax|msisdn)($|_)"],
    # ip_address precedes address so "ip_address" gets the more specific semantic type.
    "ip_address": [r"(^|_)ip($|_)", r"ip_?addr"],
    "address": [r"address", r"(^|_)(street|addr|address_line)($|_)"],
    "postcode": [r"postcode", r"postal", r"(^|_)zip($|_|code)"],
    "iban": [r"iban", r"(^|_)(swift|bic)($|_)", r"(account|card)_?(number|no|num|nr)", r"(^|_)ccn($|_)"],
    "national_id": [
        r"(^|_)(ssn|dni|nif|nie|nir|cpf|nino|passport)($|_)",
        r"social_?security",
        r"(tax|national|personal)_?id",
    ],
    "birth_date": [r"birth", r"(^|_)dob($|_)"],
    "geo": [r"(^|_)(lat|lon|lng|latitude|longitude|geo|geolocation)($|_)"],
    "secret": [r"password", r"(^|_)(secret|token|api_?key)($|_)"],
    "special_category": [r"(^|_)(ethnic|ethnicity|religion|health|diagnosis|medical)($|_)"],
}

DEFAULT_ID_NAME_PATTERNS = [r"^id$", r"_id$", r"^id_", r"_key$", r"_uuid$", r"^uuid$", r"_no$", r"_nr$", r"^sku$"]


class PrivacyConfig(_Section):
    k: int = Field(10, description=f"Minimum group size. Never below {MIN_K}.")
    top_k_values: int = Field(50, ge=1, description="Max category labels exported per column.")
    clip_lower: float = Field(0.01, description="Quantile exported as 'min'. Must be >= 0.01.")
    clip_upper: float = Field(0.99, description="Quantile exported as 'max'. Must be <= 0.99.")
    min_count_for_distribution: int = Field(
        50, description="Non-null rows needed before quantiles, mean, std or length stats are exported."
    )
    pii_name_patterns: dict[str, list[str]] = Field(default_factory=lambda: dict(DEFAULT_PII_NAME_PATTERNS))
    pii_tag_patterns: list[str] = Field(
        default_factory=lambda: ["pii", "pii.*", "pii_*", "class.*", "sensitive", "personal*", "gdpr*"],
        description="Glob patterns matched against a UC tag name and against 'name=value'.",
    )
    sensitive_columns: list[str] = Field(
        default_factory=list, description="Glob patterns on catalog.schema.table.column forced to sensitive."
    )
    value_pii_min_share: float = Field(
        0.3, gt=0, le=1, description="Share of values matching an email/phone/IP/IBAN regex that marks a column PII."
    )
    allowed_labels: list[str] = Field(
        default_factory=list,
        description="Columns (glob on catalog.schema.table.column) whose category labels may leave. Used by the guard.",
    )

    @field_validator("k")
    @classmethod
    def _k_floor(cls, v: int) -> int:
        if v < MIN_K:
            raise ValueError(f"k must be >= {MIN_K}, got {v}")
        return v

    @model_validator(mode="after")
    def _bounds(self) -> PrivacyConfig:
        if self.clip_lower < 0.01 or self.clip_upper > 0.99 or self.clip_lower >= self.clip_upper:
            raise ValueError("clip_lower must be >= 0.01, clip_upper <= 0.99, and clip_lower < clip_upper")
        if self.min_count_for_distribution < self.k:
            raise ValueError("min_count_for_distribution must be >= k")
        return self


class ProfilingConfig(_Section):
    max_categorical_distinct: int = Field(200, ge=1)
    max_categorical_ratio: float = Field(0.5, gt=0, le=1)
    high_cardinality_ratio: float = Field(
        0.5, gt=0, le=1, description="String columns above this distinct ratio export no values."
    )
    quantiles: list[float] = Field(default_factory=lambda: [0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
    percentile_accuracy: int = Field(10000, ge=100)
    approx_distinct_rsd: float = Field(0.02, gt=0, le=0.1)
    pattern_top_n: int = Field(5, ge=1)
    pattern_max_length: int = Field(40, ge=1, description="Values longer than this get no format pattern.")
    max_year_months: int = Field(600, ge=1, description="Max year-month buckets exported per date column.")
    max_decimals: int = Field(6, ge=0)
    free_text_min_mean_length: float = Field(30.0, gt=0)
    id_name_patterns: list[str] = Field(default_factory=lambda: list(DEFAULT_ID_NAME_PATTERNS))
    sample_fraction: float = Field(1.0, gt=0, le=1)
    sample_fractions: dict[str, float] = Field(
        default_factory=dict, description="Per-table override, keyed by catalog.schema.table."
    )
    sample_seed: int = 42

    @field_validator("quantiles")
    @classmethod
    def _quantiles(cls, v: list[float]) -> list[float]:
        if any(q <= 0 or q >= 1 for q in v):
            raise ValueError("quantiles must be strictly between 0 and 1")
        return sorted(set(v))

    @field_validator("sample_fractions")
    @classmethod
    def _fractions(cls, v: dict[str, float]) -> dict[str, float]:
        for name, f in v.items():
            if not 0 < f <= 1:
                raise ValueError(f"sample fraction for {name} must be in (0, 1]")
        return v


class ForcedForeignKey(_Section):
    child: str = Field(description="catalog.schema.table.column")
    parent: str = Field(description="catalog.schema.table.column")


class RelationshipsConfig(_Section):
    fk_min_containment: float = Field(0.9, gt=0, le=1)
    name_similarity_threshold: float = Field(0.8, gt=0, le=1)
    pk_prefilter_ratio: float = Field(
        0.9, gt=0, le=1, description="Approx distinct ratio a column needs before an exact uniqueness check."
    )
    max_pk_pair_columns: int = Field(6, ge=2)
    children_quantiles: list[float] = Field(default_factory=lambda: [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
    forced_pks: dict[str, list[str]] = Field(default_factory=dict, description="table -> key columns")
    forced_fks: list[ForcedForeignKey] = Field(default_factory=list)


class RulesConfig(_Section):
    date_order_min_rate: float = Field(0.9, gt=0.5, le=1)
    identity_min_rate: float = Field(0.95, gt=0.5, le=1)
    identity_sample_rows: int = Field(20000, ge=100)
    identity_abs_tolerance: float = Field(0.01, ge=0)
    identity_rel_tolerance: float = Field(1e-6, ge=0)
    max_identity_columns: int = Field(10, ge=3)


class SynthgenConfig(_Section):
    dataset: str
    source: SourceConfig
    output: OutputConfig
    metadata_provider: Literal["auto", "unity_catalog", "spark"] = "auto"
    privacy: PrivacyConfig = Field(default_factory=PrivacyConfig)
    profiling: ProfilingConfig = Field(default_factory=ProfilingConfig)
    relationships: RelationshipsConfig = Field(default_factory=RelationshipsConfig)
    rules: RulesConfig = Field(default_factory=RulesConfig)

    @classmethod
    def load(cls, path: str | Path) -> SynthgenConfig:
        import yaml  # only needed for YAML configs; notebooks configure in Python

        with open(path, encoding="utf-8") as fh:
            raw = yaml.safe_load(fh) or {}
        return cls.model_validate(raw)

    def sample_fraction_for(self, table: str) -> float:
        return self.profiling.sample_fractions.get(table, self.profiling.sample_fraction)
''')

_load("synthgen.profile_schema", r'''
"""Versioned schema for profile files: the only contract between the two workspaces.

Conventions used throughout:
- Shares of column values (categories, patterns, months, ...) are over the column's non-null values.
  `null_rate` and boolean shares are over all rows.
- A rate or share is `null` and its field name is listed in `suppressed` when the underlying count
  was between 1 and k-1. A share of exactly 0.0 means the count was 0.
- `min_clipped` / `max_clipped` are the configured clip quantiles (default 1st and 99th), never true extremes.
  Numeric mean and std are computed on values capped at those bounds, so outliers cannot inflate them.
- Quantile keys are the quantile written as a string, e.g. "0.5".

Run `python -m synthgen.profile_schema schemas/` to regenerate the JSON Schema files for reviewers.
"""

from __future__ import annotations

import sys
from datetime import datetime
from pathlib import Path
from typing import Annotated, Literal, Union

from pydantic import BaseModel, ConfigDict, Field

SCHEMA_VERSION = "1.0"

Share = Annotated[float, Field(ge=0.0, le=1.0)]

Role = Literal["key", "foreign_key", "categorical", "numeric", "date", "text", "boolean", "other"]

SemanticType = Literal[
    "email",
    "phone",
    "person_name",
    "postcode",
    "address",
    "uuid",
    "ip_address",
    "iban",
    "national_id",
    "birth_date",
    "geo",
    "secret",
    "special_category",
    "identifier",
    "free_text",
    "short_text",
    "unknown",
]


class _Model(BaseModel):
    model_config = ConfigDict(extra="forbid", validate_assignment=True, allow_inf_nan=False)


class LengthStats(_Model):
    """String length statistics. Min and max are clipped quantiles of the length."""

    min_clipped: float | None = None
    mean: float | None = None
    max_clipped: float | None = None


class ValueShare(_Model):
    value: str
    share: Share


class PatternShare(_Model):
    pattern: str = Field(description="Letters mapped to A, digits to 9, punctuation and spaces kept.")
    share: Share


class NumericStats(_Model):
    min_clipped: float | None = None
    max_clipped: float | None = None
    mean: float | None = Field(None, description="Mean of values capped at min_clipped and max_clipped.")
    std: float | None = Field(None, description="Sample std of values capped at min_clipped and max_clipped.")
    quantiles: dict[str, float] = Field(default_factory=dict)
    zero_share: Share | None = None
    negative_share: Share | None = None
    is_integer: bool
    decimals: int = Field(ge=0)


class CategoricalStats(_Model):
    top_values: list[ValueShare] = Field(default_factory=list)
    other_share: Share = Field(description="Share of non-null values in labels not listed (rare or beyond top K).")
    labels_not_exported: int = Field(ge=0, description="Approximate number of labels folded into 'other'.")
    length: LengthStats = Field(default_factory=LengthStats)


class TextStats(_Model):
    """Exported for sensitive, identifier and high-cardinality columns. Never contains values."""

    length: LengthStats = Field(default_factory=LengthStats)
    patterns: list[PatternShare] = Field(default_factory=list)
    other_pattern_share: Share
    semantic_type: SemanticType


class DateStats(_Model):
    """Date and timestamp statistics. Dates are ISO strings; UTC timestamps end in 'Z'."""

    min_clipped: str | None = None
    max_clipped: str | None = None
    quantiles: dict[str, str] = Field(default_factory=dict)
    year_month_shares: dict[str, Share] = Field(default_factory=dict)
    other_year_month_share: Share = 0.0
    day_of_week_shares: dict[str, Share] = Field(default_factory=dict)
    # Timestamp only.
    hour_shares: dict[str, Share] | None = None
    always_midnight: bool | None = None
    timezone: str | None = None


class BooleanStats(_Model):
    true_share: Share | None = None
    false_share: Share | None = None


class ColumnProfile(_Model):
    name: str
    position: int = Field(ge=0)
    type: str = Field(description="Spark SQL type string.")
    nullable: bool
    comment: str | None = None
    tags: dict[str, str] = Field(default_factory=dict)
    is_partition: bool = False
    role: Role
    sensitive: bool
    sensitive_reasons: list[str] = Field(default_factory=list)
    null_rate: Share | None = None
    distinct_approx: int = Field(ge=0)
    distinct_ratio: float | None = Field(None, ge=0.0, description="distinct_approx / non-null count.")
    suppressed: list[str] = Field(default_factory=list)
    numeric: NumericStats | None = None
    categorical: CategoricalStats | None = None
    text: TextStats | None = None
    date: DateStats | None = None
    boolean: BooleanStats | None = None


class DateOrderRule(_Model):
    kind: Literal["date_order"] = "date_order"
    before: str
    after: str
    match_rate: Share | None = Field(description="Share of rows with both set where before <= after.")
    suppressed: list[str] = Field(default_factory=list)


class NumericIdentityRule(_Model):
    kind: Literal["numeric_identity"] = "numeric_identity"
    target: str
    operation: Literal["product", "sum"]
    operands: list[str] = Field(min_length=2, max_length=2)
    match_rate: Share | None
    sample_fraction: float = Field(gt=0, le=1)
    suppressed: list[str] = Field(default_factory=list)


Rule = Annotated[Union[DateOrderRule, NumericIdentityRule], Field(discriminator="kind")]


class DeclaredForeignKey(_Model):
    name: str | None = None
    columns: list[str] = Field(min_length=1)
    parent_table: str
    parent_columns: list[str] = Field(min_length=1)


class TableProfile(_Model):
    schema_version: Literal["1.0"] = SCHEMA_VERSION
    generated_at: datetime
    profiler_version: str
    source_table: str
    row_count: int = Field(ge=0)
    sample_fraction: float = Field(gt=0, le=1)
    k_threshold: int = Field(ge=5)
    comment: str | None = None
    tags: dict[str, str] = Field(default_factory=dict)
    primary_key: list[str] | None = Field(None, description="Declared primary key, if any.")
    foreign_keys: list[DeclaredForeignKey] = Field(default_factory=list, description="Declared foreign keys.")
    partition_columns: list[str] = Field(default_factory=list)
    columns: list[ColumnProfile]
    rules: list[Rule] = Field(default_factory=list)
    warnings: list[str] = Field(default_factory=list)


KeySource = Literal["declared", "config", "inferred"]


class PrimaryKey(_Model):
    table: str
    columns: list[str] = Field(min_length=1)
    source: KeySource
    alternatives: list[list[str]] = Field(default_factory=list, description="Other unique column sets found.")


class ChildrenPerParent(_Model):
    quantiles: dict[str, float] = Field(default_factory=dict)
    zero_children_share: Share | None = None


class ForeignKey(_Model):
    child_table: str
    child_columns: list[str] = Field(min_length=1)
    parent_table: str
    parent_columns: list[str] = Field(min_length=1)
    source: KeySource
    name_score: float | None = None
    containment: Share | None = Field(None, description="Share of non-null child rows whose value is in the parent key.")
    orphan_rate: Share | None = Field(None, description="Share of non-null child rows whose value is not in the parent key.")
    orphan_value_share: Share | None = Field(
        None, description="Share of distinct non-null child values that are not in the parent key."
    )
    children_per_parent: ChildrenPerParent | None = None
    suppressed: list[str] = Field(default_factory=list)
    warnings: list[str] = Field(default_factory=list)


class Relationships(_Model):
    schema_version: Literal["1.0"] = SCHEMA_VERSION
    generated_at: datetime
    profiler_version: str
    k_threshold: int = Field(ge=5)
    tables: list[str]
    primary_keys: list[PrimaryKey] = Field(default_factory=list)
    foreign_keys: list[ForeignKey] = Field(default_factory=list)


def write_json_schemas(out_dir: str | Path) -> list[Path]:
    import json

    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    written = []
    for model, name in [(TableProfile, "table_profile"), (Relationships, "relationships")]:
        path = out / f"{name}.v{SCHEMA_VERSION}.schema.json"
        path.write_text(json.dumps(model.model_json_schema(), indent=2) + "\n", encoding="utf-8")
        written.append(path)
    return written


if __name__ == "__main__":
    for p in write_json_schemas(sys.argv[1] if len(sys.argv) > 1 else "schemas"):
        print(p)
''')

_load("synthgen.io", r'''
"""Reading and writing profile files.

All profile output goes through `ProfileStore`, which only writes inside its own root folder and
validates every file against the profile schema on write and on read.
"""

from __future__ import annotations

import json
from pathlib import Path
from typing import TypeVar

from pydantic import BaseModel

from synthgen.profile_schema import Relationships, TableProfile

REVIEW_DIR = "review"
RELATIONSHIPS_FILE = "relationships.json"
PROFILE_SUFFIX = ".profile.json"

M = TypeVar("M", bound=BaseModel)


class OutputPathError(RuntimeError):
    pass


def dump_json(model: BaseModel) -> str:
    """Human-readable, stable JSON: indented, field order fixed by the schema."""
    return json.dumps(model.model_dump(mode="json"), indent=2, ensure_ascii=False) + "\n"


class ProfileStore:
    """A folder of profile files, e.g. <output_root>/review."""

    def __init__(self, folder: str | Path):
        self.folder = Path(folder).resolve()

    @classmethod
    def review(cls, output_root: str | Path) -> ProfileStore:
        return cls(Path(output_root) / REVIEW_DIR)

    def _path(self, filename: str) -> Path:
        if "/" in filename or "\\" in filename or filename.startswith("."):
            raise OutputPathError(f"invalid profile file name: {filename!r}")
        path = (self.folder / filename).resolve()
        if path.parent != self.folder:
            raise OutputPathError(f"{path} is outside {self.folder}")
        return path

    def _write(self, filename: str, model: BaseModel) -> Path:
        text = dump_json(model)
        # Round-trip through the schema so nothing invalid is ever written.
        type(model).model_validate_json(text)
        self.folder.mkdir(parents=True, exist_ok=True)
        path = self._path(filename)
        path.write_text(text, encoding="utf-8")
        return path

    def write_table_profile(self, profile: TableProfile) -> Path:
        return self._write(profile.source_table + PROFILE_SUFFIX, profile)

    def write_relationships(self, rel: Relationships) -> Path:
        return self._write(RELATIONSHIPS_FILE, rel)

    def _read(self, filename: str, model: type[M]) -> M:
        return model.model_validate_json(self._path(filename).read_text(encoding="utf-8"))

    def read_table_profile(self, table: str) -> TableProfile:
        return self._read(table + PROFILE_SUFFIX, TableProfile)

    def read_relationships(self) -> Relationships:
        return self._read(RELATIONSHIPS_FILE, Relationships)

    def table_profiles(self) -> list[TableProfile]:
        return [
            TableProfile.model_validate_json(p.read_text(encoding="utf-8"))
            for p in sorted(self.folder.glob("*" + PROFILE_SUFFIX))
        ]
''')

_load("synthgen.profiler", r'''
''', package=True)

_load("synthgen.profiler.detect", r'''
"""Pure-Python heuristics: PII detection, identifier names, roles, semantic types, FK name matching.

Nothing here touches Spark or data; it works on names, types, tags and aggregate statistics only.
"""

from __future__ import annotations

import fnmatch
import re
from dataclasses import dataclass, field
from difflib import SequenceMatcher

from synthgen.config import SynthgenConfig

# Value regexes evaluated in Spark (Java regex syntax); shares feed semantic typing and value-based PII.
VALUE_REGEXES: dict[str, str] = {
    "email": r"^[^@\s]+@[^@\s]+\.[A-Za-z]{2,}$",
    "uuid": r"^[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}$",
    # 8-15 digits with common separators; IPv4 addresses and ISO dates excluded.
    "phone": r"^(?!([0-9]{1,3}\.){3}[0-9]{1,3}$)(?![0-9]{4}-[0-9]{2}-[0-9]{2})(?=(\D*\d){8,15}\D*$)\+?[0-9(][0-9 ().-]{6,22}[0-9]$",
    "ip_address": r"^([0-9]{1,3}\.){3}[0-9]{1,3}$|^[0-9a-fA-F:]{2,39}:[0-9a-fA-F]{1,4}$",
    "iban": r"^[A-Z]{2}[0-9]{2}[A-Z0-9]{10,30}$",
}
# Value patterns strong enough on their own to mark a column sensitive.
VALUE_PII_TYPES = ("email", "phone", "ip_address", "iban")

INTEGER_TYPES = {"tinyint", "smallint", "int", "bigint", "byte", "short", "integer", "long"}
FLOAT_TYPES = {"float", "double", "real"}


def type_family(spark_type: str) -> str:
    """Coarse type family: integer, decimal, float, string, date, timestamp, boolean, other."""
    t = spark_type.lower()
    if t in INTEGER_TYPES:
        return "integer"
    if t.startswith("decimal"):
        m = re.match(r"decimal\((\d+),\s*(\d+)\)", t)
        return "integer" if m and m.group(2) == "0" else "decimal"
    if t in FLOAT_TYPES:
        return "float"
    if t == "string" or t.startswith("varchar") or t.startswith("char"):
        return "string"
    if t == "date":
        return "date"
    if t.startswith("timestamp"):
        return "timestamp"
    if t == "boolean":
        return "boolean"
    return "other"


def is_numeric_family(family: str) -> bool:
    return family in ("integer", "decimal", "float")


def decimal_scale(spark_type: str) -> int | None:
    m = re.match(r"decimal\((\d+),\s*(\d+)\)", spark_type.lower())
    return int(m.group(2)) if m else None


def _matches_any(name: str, patterns: list[str]) -> bool:
    return any(re.search(p, name) for p in patterns)


def is_id_name(column: str, cfg: SynthgenConfig) -> bool:
    return _matches_any(column.lower(), cfg.profiling.id_name_patterns)


def pii_name_matches(column: str, cfg: SynthgenConfig) -> list[str]:
    """Semantic types whose name patterns match the column name, in config order."""
    name = column.lower()
    return [sem for sem, pats in cfg.privacy.pii_name_patterns.items() if _matches_any(name, pats)]


def pii_tag_matches(tags: dict[str, str], cfg: SynthgenConfig) -> list[str]:
    hits = []
    for name, value in tags.items():
        candidates = [name.lower(), f"{name}={value}".lower()]
        if any(fnmatch.fnmatch(c, p.lower()) for c in candidates for p in cfg.privacy.pii_tag_patterns):
            hits.append(f"{name}={value}" if value else name)
    return hits


def config_marks_sensitive(full_column: str, cfg: SynthgenConfig) -> bool:
    return any(fnmatch.fnmatch(full_column.lower(), p.lower()) for p in cfg.privacy.sensitive_columns)


@dataclass
class Sensitivity:
    sensitive: bool
    reasons: list[str] = field(default_factory=list)
    semantic_hint: str | None = None


def static_sensitivity(table: str, column: str, tags: dict[str, str], cfg: SynthgenConfig) -> Sensitivity:
    """Sensitivity known before reading data: UC tags, name heuristics, config."""
    reasons: list[str] = []
    for hit in pii_tag_matches(tags, cfg):
        reasons.append(f"uc_tag:{hit}")
    name_hits = pii_name_matches(column, cfg)
    reasons.extend(f"name:{sem}" for sem in name_hits)
    if config_marks_sensitive(f"{table}.{column}", cfg):
        reasons.append("config")
    return Sensitivity(bool(reasons), reasons, name_hits[0] if name_hits else None)


def value_pii_reasons(regex_shares: dict[str, float], id_named: bool, cfg: SynthgenConfig) -> list[str]:
    """Reasons from value patterns. Phone is skipped for identifier columns (numeric-looking ids)."""
    out = []
    for sem in VALUE_PII_TYPES:
        if sem == "phone" and id_named:
            continue
        if regex_shares.get(sem, 0.0) >= cfg.privacy.value_pii_min_share:
            out.append(f"value_pattern:{sem}")
    return out


def guess_semantic_type(
    *,
    name_hint: str | None,
    regex_shares: dict[str, float],
    id_named: bool,
    mean_length: float | None,
    space_share: float,
    cfg: SynthgenConfig,
) -> str:
    if name_hint:
        return name_hint
    best = max(regex_shares.items(), key=lambda kv: kv[1], default=(None, 0.0))
    if best[0] and best[1] >= cfg.privacy.value_pii_min_share:
        return best[0]
    if id_named:
        return "identifier"
    if mean_length is not None and mean_length >= cfg.profiling.free_text_min_mean_length and space_share >= 0.5:
        return "free_text"
    if mean_length is not None:
        return "short_text"
    return "unknown"


def string_is_categorical(distinct: int, ratio: float | None, cfg: SynthgenConfig) -> bool:
    p = cfg.profiling
    if ratio is None:
        return True  # all null: nothing to export either way
    return (
        distinct <= p.max_categorical_distinct
        and ratio <= p.max_categorical_ratio
        and ratio <= p.high_cardinality_ratio
    )


# --- foreign key name matching -------------------------------------------------------------

_KEY_SUFFIX = re.compile(r"(_?(id|key|uuid|no|nr|code))$")


def singular(word: str) -> str:
    if word.endswith("ies") and len(word) > 4:
        return word[:-3] + "y"
    if word.endswith(("ses", "xes", "ches", "shes")):
        return word[:-2]
    if word.endswith("s") and not word.endswith("ss") and len(word) > 3:
        return word[:-1]
    return word


def normalize_key_name(name: str) -> str:
    return _KEY_SUFFIX.sub("", name.lower()).strip("_")


def fk_name_score(child_column: str, parent_table: str, parent_column: str) -> float:
    """How strongly a child column name suggests a reference to parent_table.parent_column (0..1)."""
    child = child_column.lower()
    pcol = parent_column.lower()
    ptable = singular(parent_table.split(".")[-1].lower())
    stem = normalize_key_name(child)
    if child == pcol and pcol != "id":
        return 1.0
    if stem == ptable:
        return 0.95
    if stem.endswith("_" + ptable):  # e.g. referred_by_customer_id -> customers
        return 0.85
    if stem and stem == normalize_key_name(pcol):
        return 0.9
    if not stem:
        return 0.0
    return max(SequenceMatcher(None, stem, ptable).ratio(), SequenceMatcher(None, stem, normalize_key_name(pcol)).ratio())
''')

_load("synthgen.generator", r'''
''', package=True)

_load("synthgen.generator.vocab", r'''
"""Built-in word lists for realistic-looking fake text. No external packages (no Faker)."""

FIRST_NAMES = [
    "Emma", "Liam", "Olivia", "Noah", "Ava", "Lucas", "Mia", "Hugo", "Sofia", "Leo", "Chloe", "Louis", "Lea",
    "Gabriel", "Alice", "Arthur", "Julia", "Adam", "Laura", "Paul", "Marie", "Jules", "Clara", "Nathan",
    "Sarah", "Victor", "Anna", "Thomas", "Ines", "Mateo", "Lena", "Ethan", "Nina", "Oscar", "Elena", "Marco",
    "Ana", "Pablo", "Lucia", "Diego", "Carmen", "Jonas", "Hannah", "Felix", "Greta", "Omar", "Yasmine",
    "Ali", "Fatima", "Kenji", "Yuki", "Priya", "Arjun", "Mei", "Wei", "Sven", "Ingrid", "Piotr", "Zofia",
]
LAST_NAMES = [
    "Martin", "Bernard", "Dubois", "Thomas", "Robert", "Richard", "Petit", "Durand", "Leroy", "Moreau",
    "Simon", "Laurent", "Lefebvre", "Michel", "Garcia", "David", "Bertrand", "Roux", "Vincent", "Fournier",
    "Morel", "Girard", "Andre", "Mercier", "Dupont", "Lambert", "Bonnet", "Francois", "Martinez", "Legrand",
    "Smith", "Johnson", "Brown", "Miller", "Wilson", "Taylor", "Muller", "Schmidt", "Schneider", "Fischer",
    "Rossi", "Russo", "Ferrari", "Esposito", "Lopez", "Gonzalez", "Rodriguez", "Fernandez", "Silva", "Santos",
    "Jansen", "De Vries", "Nowak", "Kowalski", "Novak", "Horvat", "Nielsen", "Hansen", "Tanaka", "Kim",
]
STREETS = [
    "Main Street", "Oak Avenue", "Station Road", "Church Lane", "Park Road", "High Street", "Mill Lane",
    "rue de la Paix", "avenue des Fleurs", "boulevard Victor Hugo", "rue du Marche", "chemin des Pins",
    "Calle Mayor", "Via Roma", "Hauptstrasse", "Bahnhofstrasse", "Kerkstraat", "Lindenweg", "Harbour View",
]
CITIES = ["Springfield", "Riverton", "Lakeside", "Fairview", "Greenville", "Mapleton", "Brookfield", "Hillcrest"]
EMAIL_DOMAINS = ["example.com", "example.org", "example.net", "mail.example"]
WORDS = (
    "the a and of to in for with on this that order customer product service team delivery quality price "
    "fast great good thanks please happy gift friend family birthday wishes love best regards new item "
    "package box return refund question about request update account support store online thank you very "
    "much again soon always time day week month year nice perfect excellent small large color size"
).split()
''')

_load("synthgen.generator.plan", r'''
"""Generation plan: which tables, in which order, how many rows, which FK drives each table's size."""

from __future__ import annotations

import io
import logging
import zipfile
from dataclasses import dataclass, field
from pathlib import Path

from synthgen.io import PROFILE_SUFFIX, RELATIONSHIPS_FILE
from synthgen.profile_schema import ColumnProfile, ForeignKey, PrimaryKey, Relationships, TableProfile

log = logging.getLogger(__name__)


@dataclass
class GeneratorSettings:
    scale: float = 0.01  # multiplies every source row count
    min_rows: int = 100  # tables never shrink below this (or their real size, if smaller)
    max_rows: int = 1_000_000  # hard cap per table: rows are generated in Python on the driver
    seed: int = 42

    def __post_init__(self) -> None:
        if not 0 < self.scale <= 10:
            raise ValueError("scale must be in (0, 10]")
        if self.min_rows < 1 or self.max_rows < self.min_rows:
            raise ValueError("need 1 <= min_rows <= max_rows")


@dataclass
class TablePlan:
    profile: TableProfile
    target_rows: int
    primary_key: list[str] = field(default_factory=list)
    driving_fk: ForeignKey | None = None  # child rows are created per parent row along this FK
    foreign_keys: dict[str, ForeignKey] = field(default_factory=dict)  # child column -> FK

    @property
    def table(self) -> str:
        return self.profile.source_table

    @property
    def name(self) -> str:
        return self.table.split(".")[-1]

    def column(self, name: str) -> ColumnProfile:
        return next(c for c in self.profile.columns if c.name == name)


def load_profiles(source: str | Path) -> tuple[list[TableProfile], Relationships]:
    """Read profiles from a folder (or its review/ subfolder) or from the zip sent by the customer."""
    path = Path(source)
    files: dict[str, str] = {}
    if path.suffix == ".zip":
        with zipfile.ZipFile(path) as archive:
            for name in archive.namelist():
                if name.endswith(".json"):
                    files[Path(name).name] = archive.read(name).decode("utf-8")
    else:
        folder = path / "review" if (path / "review" / RELATIONSHIPS_FILE).exists() else path
        files = {p.name: p.read_text(encoding="utf-8") for p in folder.glob("*.json")}
    if RELATIONSHIPS_FILE not in files:
        raise FileNotFoundError(f"{RELATIONSHIPS_FILE} not found in {source}")
    profiles = [TableProfile.model_validate_json(t) for n, t in sorted(files.items()) if n.endswith(PROFILE_SUFFIX)]
    if not profiles:
        raise FileNotFoundError(f"no *{PROFILE_SUFFIX} files in {source}")
    return profiles, Relationships.model_validate_json(files[RELATIONSHIPS_FILE])


def _scaled(row_count: int, s: GeneratorSettings) -> int:
    return min(row_count, s.max_rows, max(s.min_rows, round(row_count * s.scale))) if row_count else 0


def _choose_driving_fk(plan: TablePlan, fks: list[ForeignKey], sizes: dict[str, int]) -> ForeignKey | None:
    candidates = [
        fk for fk in fks
        if fk.parent_table != plan.table and len(fk.child_columns) == 1 and fk.children_per_parent
        and fk.children_per_parent.quantiles and fk.parent_table in sizes
    ]
    if not candidates:
        return None
    # Prefer an FK that is part of the child's key (order_lines.order_id), then the biggest parent.
    return max(candidates, key=lambda fk: (fk.child_columns[0] in plan.primary_key, sizes[fk.parent_table]))


def build_plan(profiles: list[TableProfile], rel: Relationships, settings: GeneratorSettings) -> list[TablePlan]:
    """Tables in generation order: every driving parent comes before its children."""
    pks: dict[str, PrimaryKey] = {pk.table: pk for pk in rel.primary_keys}
    by_table = {p.source_table: p for p in profiles}
    fks_of: dict[str, list[ForeignKey]] = {}
    for fk in rel.foreign_keys:
        if fk.child_table in by_table and fk.parent_table in by_table:
            fks_of.setdefault(fk.child_table, []).append(fk)

    plans = {
        t: TablePlan(
            profile=p,
            target_rows=_scaled(p.row_count, settings),
            primary_key=list(pks[t].columns) if t in pks else [],
            foreign_keys={fk.child_columns[0]: fk for fk in fks_of.get(t, []) if len(fk.child_columns) == 1},
        )
        for t, p in by_table.items()
    }

    def parents(t: str) -> set[str]:
        return {fk.parent_table for fk in fks_of.get(t, []) if fk.parent_table != t}

    ordered: list[TablePlan] = []
    done: dict[str, int] = {}
    pending = sorted(plans)
    while pending:
        ready = [t for t in pending if parents(t) <= set(done)] or pending[:1]  # pending[:1] breaks a cycle
        for t in ready:
            plan = plans[t]
            plan.driving_fk = _choose_driving_fk(plan, fks_of.get(t, []), done)
            ordered.append(plan)
            done[t] = plan.target_rows
            pending.remove(t)
    return ordered


def describe_plan(plans: list[TablePlan], settings: GeneratorSettings) -> str:
    out = io.StringIO()
    out.write(f"scale={settings.scale:g}  min_rows={settings.min_rows}  max_rows={settings.max_rows}  seed={settings.seed}\n")
    for i, p in enumerate(plans, 1):
        size = (
            f"rows from {p.driving_fk.parent_table.split('.')[-1]}.{p.driving_fk.parent_columns[0]} "
            f"(children per parent)"
            if p.driving_fk else f"~{p.target_rows} rows (source {p.profile.row_count})"
        )
        links = ", ".join(f"{c} -> {fk.parent_table.split('.')[-1]}.{fk.parent_columns[0]}"
                          for c, fk in p.foreign_keys.items())
        out.write(f"{i}. {p.name}: {size}; key {p.primary_key or '-'}; links: {links or '-'}\n")
    return out.getvalue()
''')

_load("synthgen.generator.columns", r'''
"""Value samplers, one per kind of column profile. Pure Python, deterministic for a given Random."""

from __future__ import annotations

import bisect
import calendar
import random
import re
import string
import uuid
from datetime import date, datetime, timedelta
from decimal import ROUND_HALF_UP, Decimal
from zoneinfo import ZoneInfo

from synthgen.generator import vocab
from synthgen.profile_schema import ColumnProfile, DateStats, TextStats
from synthgen.profiler.detect import decimal_scale, type_family

DOW = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]


def rng_for(seed: int, *parts: object) -> random.Random:
    """Independent, reproducible stream per (seed, table, column, purpose)."""
    return random.Random(":".join(str(p) for p in (seed, *parts)))


# --- distributions ---------------------------------------------------------------------------


def quantile_points(quantiles: dict[str, float]) -> list[tuple[float, float]]:
    return sorted((float(q), float(v)) for q, v in quantiles.items())


def stratified_uniforms(n: int, rng: random.Random) -> list[float]:
    """n uniforms, one per equal-width stratum, shuffled: quantiles are reproduced even for small n."""
    us = [(i + rng.random()) / n for i in range(n)]
    rng.shuffle(us)
    return us


def inverse_cdf(points: list[tuple[float, float]], u: float) -> float:
    """Linear interpolation through (probability, value) points; clamps outside the first/last point."""
    if u <= points[0][0]:
        return points[0][1]
    if u >= points[-1][0]:
        return points[-1][1]
    i = bisect.bisect_right([p for p, _ in points], u)
    (p0, v0), (p1, v1) = points[i - 1], points[i]
    return v0 + (v1 - v0) * (u - p0) / (p1 - p0)


class Weighted:
    """Fast repeated weighted choice."""

    def __init__(self, items: list, weights: list[float]):
        total = sum(weights)
        if not items or total <= 0:
            raise ValueError("Weighted needs items with a positive total weight")
        self.items, self.cum, acc = items, [], 0.0
        for w in weights:
            acc += w
            self.cum.append(acc)

    def pick(self, rng: random.Random):
        return self.at(rng.random())

    def at(self, u: float):
        """The item at cumulative position u in [0, 1): use with stratified uniforms for exact shares."""
        return self.items[min(bisect.bisect_right(self.cum, u * self.cum[-1]), len(self.items) - 1)]


def null_mask(n: int, null_rate: float | None, rng: random.Random) -> list[bool]:
    rate = null_rate or 0.0  # a suppressed rate means fewer than k nulls: treat as none
    return [rng.random() < rate for _ in range(n)]


# --- numeric, categorical, boolean -----------------------------------------------------------


def step_inverse_cdf(points: list[tuple[float, float]], u: float) -> float:
    """Discrete version: the value of the quantile point nearest to u, so only observed values appear
    and each gets roughly the probability mass between its neighbours' midpoints."""
    probs = [p for p, _ in points]
    i = bisect.bisect_left(probs, u)
    if i <= 0:
        return points[0][1]
    if i >= len(points):
        return points[-1][1]
    return points[i - 1][1] if u - probs[i - 1] < probs[i] - u else points[i][1]


# Numeric columns with at most this many distinct values (quantity, fee tiers, ratings) are sampled
# from their quantile values only, so no in-between values appear (no fee of 7.30 between 4.99 and 9.99).
DISCRETE_MAX_DISTINCT = 20


def numeric_values(col: ColumnProfile, n: int, rng: random.Random) -> list[float]:
    s = col.numeric
    points = quantile_points(s.quantiles) if s and s.quantiles else []
    sample = step_inverse_cdf if col.distinct_approx <= DISCRETE_MAX_DISTINCT else inverse_cdf
    out = []
    for u in stratified_uniforms(n, rng):
        if points:
            v = sample(points, u)
        elif s and s.mean is not None:
            v = rng.gauss(s.mean, s.std or abs(s.mean) * 0.1 or 1.0)
        else:
            v = rng.uniform(0, 100)
        out.append(int(round(v)) if (s is None or s.is_integer) else round(v, s.decimals))
    return out


def _title(column: str) -> str:
    return column.replace("_", " ").strip().title() or "Value"


def categorical_values(col: ColumnProfile, n: int, rng: random.Random) -> list[str]:
    c = col.categorical
    items = [v.value for v in c.top_values]
    weights = [v.share for v in c.top_values]
    if c.other_share > 0 or not items:
        # Labels below k were never exported: invent stand-ins sharing the "other" share.
        count = max(1, min(c.labels_not_exported or 1, 50))
        title = _title(col.name)
        invented = [f"{title} {chr(65 + i)}" if count <= 26 else f"{title} {i + 1:02d}" for i in range(count)]
        items += invented
        weights += [max(c.other_share, 1e-9) / count] * count
    w = Weighted(items, weights)
    return [w.at(u) for u in stratified_uniforms(n, rng)]


def boolean_values(col: ColumnProfile, n: int, rng: random.Random) -> list[bool]:
    t, f = (col.boolean.true_share or 0.0), (col.boolean.false_share or 0.0)
    p = t / (t + f) if t + f > 0 else 0.5
    return [u < p for u in stratified_uniforms(n, rng)]


# --- dates ------------------------------------------------------------------------------------


def _to_date(text: str) -> date:
    return date.fromisoformat(text[:10])


class DateSampler:
    def __init__(self, stats: DateStats, timestamp: bool):
        self.s, self.timestamp = stats, timestamp
        # Hours were measured in the source session time zone; aware datetimes keep them there when
        # Spark converts (naive ones would be read in the driver's local zone). NTZ stays naive.
        self.tz = ZoneInfo(stats.timezone) if timestamp and stats.timezone else None
        self.points = [(p, _to_date(v).toordinal()) for p, v in quantile_points_str(stats.quantiles)]
        self.lo = _to_date(stats.min_clipped).toordinal() if stats.min_clipped else None
        self.hi = _to_date(stats.max_clipped).toordinal() if stats.max_clipped else None
        dow = stats.day_of_week_shares
        self.dow_weight = [dow.get(d, 0.0) for d in DOW] if dow else [1.0] * 7
        if sum(self.dow_weight) <= 0:
            self.dow_weight = [1.0] * 7
        months = stats.year_month_shares
        self.months = Weighted(list(months), list(months.values())) if months and stats.other_year_month_share <= 0.3 else None
        hours = stats.hour_shares or {}
        self.hours = Weighted([int(h) for h in hours], list(hours.values())) if hours and sum(hours.values()) > 0 else None
        self._month_days: dict[str, Weighted] = {}

    def _day_in_month(self, ym: str, rng: random.Random) -> date:
        if ym not in self._month_days:
            y, m = int(ym[:4]), int(ym[5:7])
            days = [date(y, m, d) for d in range(1, calendar.monthrange(y, m)[1] + 1)]
            if self.lo is not None and self.hi is not None:
                days = [d for d in days if self.lo <= d.toordinal() <= self.hi] or days
            self._month_days[ym] = Weighted(days, [self.dow_weight[d.weekday()] + 1e-9 for d in days])
        return self._month_days[ym].pick(rng)

    def _day(self, rng: random.Random) -> date:
        if self.months:
            return self._day_in_month(self.months.pick(rng), rng)
        top = max(self.dow_weight)
        for _ in range(20):  # rejection sampling towards the weekday profile
            if self.points:
                d = date.fromordinal(int(round(inverse_cdf(self.points, rng.random()))))
            else:
                d = date.today() - timedelta(days=rng.randint(0, 3 * 365))
            if rng.random() <= self.dow_weight[d.weekday()] / top:
                return d
        return d

    def sample(self, rng: random.Random):
        d = self._day(rng)
        if not self.timestamp:
            return d
        if self.s.always_midnight:
            return datetime(d.year, d.month, d.day, tzinfo=self.tz)
        hour = self.hours.pick(rng) if self.hours else rng.randint(0, 23)
        return datetime(d.year, d.month, d.day, hour, rng.randint(0, 59), rng.randint(0, 59), tzinfo=self.tz)


def quantile_points_str(quantiles: dict[str, str]) -> list[tuple[float, str]]:
    return sorted((float(q), v) for q, v in quantiles.items())


def date_values(col: ColumnProfile, n: int, rng: random.Random) -> list:
    sampler = DateSampler(col.date, timestamp=type_family(col.type) == "timestamp")
    return [sampler.sample(rng) for _ in range(n)]


# --- text -------------------------------------------------------------------------------------

_RUN = re.compile(r"A+|9+")
_WORDS_BY_LEN: dict[int, list[str]] = {}
for _w in vocab.WORDS:
    _WORDS_BY_LEN.setdefault(len(_w), []).append(_w)


def fill_pattern(pattern: str, rng: random.Random) -> str:
    """A -> random capital letter, 9 -> random digit, anything else kept."""
    return "".join(
        rng.choice(string.ascii_uppercase) if ch == "A" else str(rng.randint(0, 9)) if ch == "9" else ch
        for ch in pattern
    )


def _word(length: int, rng: random.Random) -> str:
    for delta in range(0, 10):
        for size in (length - delta, length + delta):
            if size in _WORDS_BY_LEN:
                return rng.choice(_WORDS_BY_LEN[size])
    return rng.choice(vocab.WORDS)


def wordy_pattern(pattern: str, rng: random.Random) -> str:
    """Like fill_pattern, but letter runs become real words: 'AAAAA AAAA AAA-999' -> 'Great Item Box-417'."""
    def repl(m: re.Match) -> str:
        run = m.group(0)
        if run[0] == "9":
            return "".join(str(rng.randint(0, 9)) for _ in run)
        return _word(len(run), rng).capitalize() if len(run) > 1 else rng.choice(string.ascii_uppercase)
    return _RUN.sub(repl, pattern)


def counter_pattern(pattern: str, i: int, letters: str) -> str:
    """Key i rendered in the shape of pattern: the last digit run becomes a zero-padded counter,
    letters are taken from `letters` (e.g. 'A-99999' + 'PRODUCTS' -> 'P-00042')."""
    runs = list(re.finditer(r"9+", pattern))
    letters = (letters.upper() or "K")
    out, li = [], 0
    for ch in pattern:
        if ch == "A":
            out.append(letters[li % len(letters)])
            li += 1
        else:
            out.append(ch)
    text = "".join(out)
    if not runs:
        return f"{text}{i}"
    run = runs[-1]
    width = run.end() - run.start()
    return text[: run.start()] + str(i).zfill(width) + text[run.end():].replace("9", "0")


def _pattern_picker(text: TextStats | None) -> Weighted | None:
    if not text or not text.patterns:
        return None
    return Weighted([p.pattern for p in text.patterns], [p.share for p in text.patterns])


def _target_length(text: TextStats | None, rng: random.Random, default: int = 12) -> int:
    if not text or text.length.mean is None:
        return default
    lo = int(text.length.min_clipped or 1)
    hi = max(lo, int(text.length.max_clipped or lo))
    drawn = int(round(rng.gauss(text.length.mean, (hi - lo) / 4 or 1)))
    return max(1, lo, min(hi, drawn))


def _sentence(length: int, rng: random.Random) -> str:
    words: list[str] = []
    while len(" ".join(words)) < length:
        words.append(rng.choice(vocab.WORDS))
    return " ".join(words).capitalize()[: max(length, 1)].rstrip() + "."


def _person(column: str, rng: random.Random) -> str:
    name = column.lower()
    if re.search(r"first|given|forename", name):
        return rng.choice(vocab.FIRST_NAMES)
    if re.search(r"last|family|surname|maiden", name):
        return rng.choice(vocab.LAST_NAMES)
    return f"{rng.choice(vocab.FIRST_NAMES)} {rng.choice(vocab.LAST_NAMES)}"


def text_value(col: ColumnProfile, rng: random.Random, patterns: Weighted | None, serial: int) -> str:
    sem = col.text.semantic_type if col.text else "unknown"
    pattern = patterns.pick(rng) if patterns else None
    if sem == "person_name":
        return _person(col.name, rng)
    if sem == "email":
        first, last = rng.choice(vocab.FIRST_NAMES), rng.choice(vocab.LAST_NAMES).replace(" ", "")
        return f"{first}.{last}{serial}@{rng.choice(vocab.EMAIL_DOMAINS)}".lower()
    if sem == "address":
        street = f"{rng.randint(1, 250)} {rng.choice(vocab.STREETS)}"
        return f"{street}, {rng.choice(vocab.CITIES)}" if pattern and "," in pattern else street
    if sem == "ip_address":
        return ".".join(str(rng.randint(1, 254)) for _ in range(4))
    if sem == "uuid":
        return str(uuid.UUID(int=rng.getrandbits(128), version=4))
    if sem == "birth_date":
        return (date(1940, 1, 1) + timedelta(days=rng.randint(0, 65 * 365))).isoformat()
    if sem == "free_text":
        return _sentence(_target_length(col.text, rng, 60), rng)
    if sem == "short_text" and not col.sensitive:  # e.g. product names; sensitive codes stay code-like
        return wordy_pattern(pattern, rng) if pattern else _sentence(_target_length(col.text, rng), rng).rstrip(".")
    if pattern:  # phone, postcode, iban, national_id, identifier, ...
        return fill_pattern(pattern, rng)
    return fill_pattern("A" * _target_length(col.text, rng, 8), rng)


def text_values(col: ColumnProfile, n: int, rng: random.Random) -> list[str]:
    """Respects the observed number of distinct values: draws from a pool when values repeat."""
    patterns = _pattern_picker(col.text)
    ratio = col.distinct_ratio if col.distinct_ratio is not None else 1.0
    pool_size = max(1, round(ratio * n))
    if pool_size >= 0.9 * n:
        return [text_value(col, rng, patterns, i + 1) for i in range(n)]
    pool = [text_value(col, rng, patterns, i + 1) for i in range(pool_size)]
    return [rng.choice(pool) for _ in range(n)]


def key_values(col: ColumnProfile, n: int, table_name: str, offset: int = 0) -> list:
    """Unique keys 1..n shaped like the source keys (digit count, prefix letters)."""
    family = type_family(col.type)
    pattern = col.text.patterns[0].pattern if col.text and col.text.patterns else None
    if family == "integer":
        digits = pattern.count("9") if pattern and set(pattern) <= {"9", "-"} else 1
        start = 10 ** (digits - 1) if digits > 1 else 1
        return [start + offset + i for i in range(n)]
    if col.text and col.text.semantic_type == "uuid":
        rng = random.Random(f"{table_name}:{col.name}:{offset}")
        return [str(uuid.UUID(int=rng.getrandbits(128), version=4)) for _ in range(n)]
    letters = re.sub(r"[^A-Za-z]", "", table_name) or "K"
    return [counter_pattern(pattern or "A-99999", offset + i + 1, letters) for i in range(n)]


# --- final typing -------------------------------------------------------------------------------


def to_spark_value(value, spark_type: str):
    """Convert a generated Python value to what Spark expects for the column type."""
    if value is None:
        return None
    family = type_family(spark_type)
    try:
        if family == "integer":
            if isinstance(value, str):
                digits = re.sub(r"\D", "", value)
                return int(digits[-15:]) if digits else None
            return int(round(float(value)))
        if family == "decimal":
            scale = decimal_scale(spark_type) or 0
            precision = int(re.match(r"decimal\((\d+)", spark_type.lower()).group(1))
            limit = 10 ** (precision - scale) - 10 ** (-scale)
            v = max(-limit, min(limit, float(value)))
            return Decimal(str(v)).quantize(Decimal(1).scaleb(-scale), rounding=ROUND_HALF_UP)
        if family == "float":
            return float(value)
        if family == "boolean":
            return bool(value)
        if family == "date":
            if isinstance(value, datetime):
                return value.date()
            return value if isinstance(value, date) else date.fromisoformat(str(value)[:10])
        if family == "timestamp":
            if isinstance(value, datetime):
                return value
            if isinstance(value, date):
                return datetime(value.year, value.month, value.day)
            return datetime.fromisoformat(str(value).replace("Z", ""))
        if family == "string":
            return value.isoformat() if isinstance(value, (date, datetime)) else str(value)
    except (ValueError, TypeError, ArithmeticError):
        return None
    return None  # complex types (arrays, maps, structs) are left empty
''')

_load("synthgen.generator.rules", r'''
"""Post-processing: make the generated rows obey the rules detected in the source."""

from __future__ import annotations

import random
import re
from datetime import date, datetime, timedelta

from synthgen.profile_schema import DateOrderRule, NumericIdentityRule, TableProfile


def _as_date(v) -> date:
    return v.date() if isinstance(v, datetime) else v


def _median_gap_days(profile: TableProfile, before: str, after: str) -> float:
    cols = {c.name: c for c in profile.columns}
    try:
        b = cols[before].date.quantiles["0.5"][:10]
        a = cols[after].date.quantiles["0.5"][:10]
        return (date.fromisoformat(a) - date.fromisoformat(b)).days
    except (AttributeError, KeyError, TypeError, ValueError):
        return 1.0


def apply_date_order(rule: DateOrderRule, profile: TableProfile, data: dict[str, list], rng: random.Random) -> None:
    """after = before + gap for a match_rate share of rows. The gap is drawn up to twice the observed
    difference between the two medians, so e.g. ship dates land a few days after order dates."""
    before, after = data.get(rule.before), data.get(rule.after)
    if before is None or after is None:
        return
    rate = 1.0 if rule.match_rate is None else rule.match_rate
    max_gap = max(1, int(round(2 * max(0.0, _median_gap_days(profile, rule.before, rule.after)))))
    for i, (b, a) in enumerate(zip(before, after)):
        if b is None or a is None or rng.random() >= rate:
            continue
        gap = timedelta(days=rng.randint(0, max_gap))
        if isinstance(a, datetime):
            base = b if isinstance(b, datetime) else datetime(b.year, b.month, b.day, tzinfo=a.tzinfo)
            moved = base + gap
            after[i] = moved if moved >= base else base
        else:
            after[i] = _as_date(b) + gap


def apply_identity(rule: NumericIdentityRule, profile: TableProfile, data: dict[str, list], rng: random.Random) -> None:
    target, (x, y) = rule.target, rule.operands
    if target not in data or x not in data or y not in data:
        return
    col = next(c for c in profile.columns if c.name == target)
    decimals = col.numeric.decimals if col.numeric else 2
    integer = bool(col.numeric and col.numeric.is_integer)
    rate = 1.0 if rule.match_rate is None else rule.match_rate
    values = data[target]
    for i, (a, b) in enumerate(zip(data[x], data[y])):
        if a is None or b is None or values[i] is None or rng.random() >= rate:
            continue
        v = a * b if rule.operation == "product" else a + b
        values[i] = int(round(v)) if integer else round(v, decimals)


def align_emails(profile: TableProfile, data: dict[str, list]) -> None:
    """Emails built from the same row's generated first and last name, when the table has both."""
    sem = {c.name: c.text.semantic_type for c in profile.columns if c.text}
    first = next((c for c, s in sem.items() if s == "person_name" and re.search(r"first|given|forename", c)), None)
    last = next((c for c, s in sem.items() if s == "person_name" and re.search(r"last|family|surname", c)), None)
    if not (first and last):
        return
    for column in (c for c, s in sem.items() if s == "email"):
        for i, email in enumerate(data[column]):
            f, la = data[first][i], data[last][i]
            if email and f and la:
                local = re.sub(r"[^a-z0-9.]", "", f"{f}.{la}".lower())
                data[column][i] = f"{local}{i + 1}@{email.split('@', 1)[1]}"


def apply_rules(profile: TableProfile, data: dict[str, list], seed: int) -> None:
    align_emails(profile, data)
    for i, rule in enumerate(profile.rules):
        rng = random.Random(f"{seed}:{profile.source_table}:rule{i}")
        if isinstance(rule, DateOrderRule):
            apply_date_order(rule, profile, data, rng)
        elif isinstance(rule, NumericIdentityRule):
            apply_identity(rule, profile, data, rng)
''')

_load("synthgen.generator.generate", r'''
"""Generate all tables in memory as columns of Python values (no Spark needed until writing)."""

from __future__ import annotations

import logging

from synthgen.generator import columns as C
from synthgen.generator.plan import GeneratorSettings, TablePlan
from synthgen.generator.rules import apply_rules
from synthgen.profile_schema import ColumnProfile, ForeignKey, TableProfile

log = logging.getLogger(__name__)

GeneratedTables = dict[str, dict[str, list]]  # table -> column -> values (one per row)


def _children_counts(fk: ForeignKey, n_parents: int, rng) -> list[int]:
    points = C.quantile_points(fk.children_per_parent.quantiles)
    return [max(0, int(round(C.inverse_cdf(points, u)))) for u in C.stratified_uniforms(n_parents, rng)]


def _dangling(parent_col: ColumnProfile, parent_keys: list, parent_table: str, m: int) -> list:
    """Keys that look like parent keys but do not exist in the parent (orphans)."""
    if m <= 0:
        return []
    if parent_keys and isinstance(parent_keys[0], int):
        top = max(parent_keys)
        return [top + 1 + i for i in range(m)]
    return C.key_values(parent_col, m, parent_table.split(".")[-1], offset=len(parent_keys) + 1000)


def _parent_column(profiles: dict[str, TableProfile], fk: ForeignKey) -> ColumnProfile:
    return next(c for c in profiles[fk.parent_table].columns if c.name == fk.parent_columns[0])


def _keys(plan: TablePlan, data: GeneratedTables, profiles: dict[str, TableProfile], s: GeneratorSettings) -> dict[str, list]:
    cols: dict[str, list] = {}
    fk = plan.driving_fk
    if fk:
        rng = C.rng_for(s.seed, plan.table, "rows")
        parent_keys = data[fk.parent_table][fk.parent_columns[0]]
        child, seq = [], []
        for key, count in zip(parent_keys, _children_counts(fk, len(parent_keys), rng)):
            child += [key] * count
            seq += range(1, count + 1)
        orphans = round((fk.orphan_rate or 0.0) * len(child))
        child += _dangling(_parent_column(profiles, fk), parent_keys, fk.parent_table, orphans)
        seq += [1] * orphans
        if len(child) > s.max_rows:
            log.warning("%s: %d rows capped to max_rows=%d", plan.name, len(child), s.max_rows)
            child, seq = child[: s.max_rows], seq[: s.max_rows]
        cols[fk.child_columns[0]] = child
        others = [c for c in plan.primary_key if c != fk.child_columns[0]]
        if others and fk.child_columns[0] in plan.primary_key:
            cols[others[0]] = seq  # e.g. order_lines.line_no: 1..k within each order
    n = len(next(iter(cols.values()))) if cols else plan.target_rows
    for key in plan.primary_key:
        if key not in cols:
            cols[key] = C.key_values(plan.column(key), n, plan.name)
    cols["__n__"] = [n]
    return cols


def _fk_values(fk: ForeignKey, data: GeneratedTables, profiles: dict[str, TableProfile], n: int, rng) -> list:
    parent_keys = data[fk.parent_table][fk.parent_columns[0]]
    # Popularity per parent follows the observed children-per-parent distribution; parents drawn
    # with weight 0 get no children, which reproduces the share of parents without children.
    weights = [float(w) for w in _children_counts(fk, len(parent_keys), rng)] if fk.children_per_parent and \
        fk.children_per_parent.quantiles else [1.0] * len(parent_keys)
    if sum(weights) <= 0:
        weights = [1.0] * len(parent_keys)
    picker = C.Weighted(parent_keys, weights)
    orphan_rate = fk.orphan_rate or 0.0
    orphan_pool = _dangling(_parent_column(profiles, fk), parent_keys, fk.parent_table, max(1, round(orphan_rate * n))) if orphan_rate else []
    return [rng.choice(orphan_pool) if orphan_pool and rng.random() < orphan_rate else picker.pick(rng) for _ in range(n)]


def _column_values(
    plan: TablePlan, col: ColumnProfile, data: GeneratedTables, profiles: dict[str, TableProfile], n: int, seed: int
) -> list:
    rng = C.rng_for(seed, plan.table, col.name)
    fk = plan.foreign_keys.get(col.name)
    if fk and fk.parent_table in data:
        values = _fk_values(fk, data, profiles, n, rng)
    elif col.numeric:
        values = C.numeric_values(col, n, rng)
    elif col.categorical:
        values = C.categorical_values(col, n, rng)
    elif col.date:
        values = C.date_values(col, n, rng)
    elif col.boolean:
        values = C.boolean_values(col, n, rng)
    elif col.text:
        values = C.text_values(col, n, rng)
    else:
        return [None] * n  # unsupported type or no statistics
    nulls = C.null_mask(n, col.null_rate, C.rng_for(seed, plan.table, col.name, "nulls"))
    return [None if is_null else v for v, is_null in zip(values, nulls)]


def generate(plans: list[TablePlan], settings: GeneratorSettings) -> GeneratedTables:
    data: GeneratedTables = {}
    profiles = {p.table: p.profile for p in plans}
    for plan in plans:  # phase 1: row counts and keys, parents before children
        data[plan.table] = _keys(plan, data, profiles, settings)
    for plan in plans:  # phase 2: every other column (FKs can now reference any table's keys)
        table = data[plan.table]
        n = table.pop("__n__")[0]
        for col in plan.profile.columns:
            if col.name not in table:
                table[col.name] = _column_values(plan, col, data, profiles, n, settings.seed)
        apply_rules(plan.profile, table, settings.seed)
        data[plan.table] = {c.name: table[c.name] for c in plan.profile.columns}
        log.info("generated %s: %d rows", plan.name, n)
    return data
''')

_load("synthgen.generator.write", r'''
"""Write generated tables to the target catalog: same names, column types, comments; keys declared."""

from __future__ import annotations

import logging
from dataclasses import dataclass, field

from pyspark.sql import SparkSession

from synthgen.generator.columns import to_spark_value
from synthgen.generator.generate import GeneratedTables
from synthgen.generator.plan import GeneratorSettings, TablePlan

log = logging.getLogger(__name__)

SOURCE_PROPERTY = "synthgen.source_table"
CHUNK_ROWS = 200_000


class TargetTableError(RuntimeError):
    pass


@dataclass
class WriteReport:
    tables: dict[str, int] = field(default_factory=dict)  # target table -> rows
    warnings: list[str] = field(default_factory=list)


def _q(*parts: str) -> str:
    return ".".join("`" + p.replace("`", "``") + "`" for p in parts)


def _lit(text: str) -> str:
    return "'" + text.replace("\\", "\\\\").replace("'", "\\'") + "'"


def target_names(plans: list[TablePlan], catalog: str, schema: str) -> dict[str, tuple[str, str, str]]:
    """Source table -> (catalog, schema, name). Same short name in two source schemas gets a prefix."""
    names = [p.name for p in plans]
    out = {}
    for p in plans:
        name = p.name if names.count(p.name) == 1 else f"{p.table.split('.')[1]}_{p.name}"
        out[p.table] = (catalog, schema, name)
    return out


def _created_by_synthgen(spark: SparkSession, table: str) -> bool:
    rows = spark.sql(f"SHOW TBLPROPERTIES {table}").collect()
    return any(r[0] == SOURCE_PROPERTY for r in rows)


def write_tables(
    spark: SparkSession,
    plans: list[TablePlan],
    data: GeneratedTables,
    catalog: str,
    schema: str,
    settings: GeneratorSettings,
    *,
    table_format: str = "delta",
    declare_keys: bool = True,
    replace_existing: bool = False,
) -> WriteReport:
    report = WriteReport()
    names = target_names(plans, catalog, schema)
    try:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {_q(catalog, schema)}")
    except Exception as exc:  # noqa: BLE001 - schema may exist without CREATE permission
        report.warnings.append(f"could not create schema {catalog}.{schema}: {exc}".splitlines()[0])

    # Refuse to drop tables that this tool did not create, before touching anything.
    for plan in plans:
        target = _q(*names[plan.table])
        if spark.catalog.tableExists(".".join(names[plan.table])) and not replace_existing \
                and not _created_by_synthgen(spark, target):
            raise TargetTableError(f"{'.'.join(names[plan.table])} exists and was not created by synthgen; "
                                   "pick another schema or set REPLACE_EXISTING = True")

    for plan in plans:
        target = _q(*names[plan.table])
        key = set(plan.primary_key) if declare_keys else set()
        cols = ",\n  ".join(
            f"{_q(c.name)} {c.type}{' NOT NULL' if c.name in key else ''}"
            + (f" COMMENT {_lit(c.comment)}" if c.comment else "")
            for c in plan.profile.columns
        )
        props = {
            SOURCE_PROPERTY: plan.table,
            "synthgen.profile_version": plan.profile.schema_version,
            "synthgen.seed": str(settings.seed),
            "synthgen.scale": f"{settings.scale:g}",
        }
        spark.sql(f"DROP TABLE IF EXISTS {target}")
        spark.sql(
            f"CREATE TABLE {target} (\n  {cols}\n) USING {table_format}"
            + (f"\nCOMMENT {_lit(plan.profile.comment)}" if plan.profile.comment else "")
            + "\nTBLPROPERTIES (" + ", ".join(f"{_lit(k)} = {_lit(v)}" for k, v in props.items()) + ")"
        )
        schema_struct = spark.table(target).schema
        types = [c.type for c in plan.profile.columns]
        values = data[plan.table]
        n = len(values[plan.profile.columns[0].name]) if plan.profile.columns else 0
        column_lists = [values[c.name] for c in plan.profile.columns]
        for start in range(0, n, CHUNK_ROWS):
            rows = [
                tuple(to_spark_value(col[i], t) for col, t in zip(column_lists, types))
                for i in range(start, min(n, start + CHUNK_ROWS))
            ]
            spark.createDataFrame(rows, schema=schema_struct).write.insertInto(target)
        report.tables[".".join(names[plan.table])] = n
        log.info("wrote %s: %d rows", ".".join(names[plan.table]), n)

    if declare_keys:
        for plan in plans:
            target = _q(*names[plan.table])
            name = names[plan.table][2]
            statements = []
            if plan.primary_key:
                statements.append(f"ALTER TABLE {target} ADD CONSTRAINT {_q(name + '_pk')} "
                                  f"PRIMARY KEY ({', '.join(_q(c) for c in plan.primary_key)})")
            for column, fk in plan.foreign_keys.items():
                parent = names.get(fk.parent_table)
                if parent:
                    statements.append(
                        f"ALTER TABLE {target} ADD CONSTRAINT {_q(f'{name}_{column}_fk')} FOREIGN KEY ({_q(column)}) "
                        f"REFERENCES {_q(*parent)} ({_q(fk.parent_columns[0])})"
                    )
            for sql in statements:
                try:
                    spark.sql(sql)
                except Exception as exc:  # noqa: BLE001 - constraints are informational; keep going
                    report.warnings.append(f"{sql.split(' ADD ')[1][:80]}: {str(exc).splitlines()[0][:160]}")
    return report
''')

_load("synthgen.generator.check", r'''
"""Quick checks on the written synthetic tables: row counts and how well the links between tables join."""

from __future__ import annotations

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from synthgen.generator.plan import TablePlan
from synthgen.generator.write import target_names


def check_tables(spark: SparkSession, plans: list[TablePlan], catalog: str, schema: str) -> list[str]:
    names = {src: ".".join(parts) for src, parts in target_names(plans, catalog, schema).items()}
    lines = []
    for plan in plans:
        lines.append(f"{names[plan.table]}: {spark.table(names[plan.table]).count()} rows")
    for plan in plans:
        for column, fk in plan.foreign_keys.items():
            if fk.parent_table not in names:
                continue
            child = spark.table(names[plan.table]).select(F.col(f"`{column}`").alias("k")).where(F.col("k").isNotNull())
            parent = spark.table(names[fk.parent_table]).select(F.col(f"`{fk.parent_columns[0]}`").alias("k"))
            total = child.count()
            joined = total - child.join(parent, "k", "left_anti").count()
            share = 100.0 * joined / total if total else 100.0
            expected = 100.0 * (1 - (fk.orphan_rate or 0.0))
            lines.append(f"{plan.name}.{column} -> {fk.parent_table.split('.')[-1]}.{fk.parent_columns[0]}: "
                         f"{share:.1f}% of rows join (source: {expected:.1f}%)")
    return lines
''')

print("synthgen engine loaded:", sys.modules["synthgen"].__version__)

In [ ]:
# ============================== 4. PLAN: what will be generated (nothing written yet) ============
import logging

from synthgen.generator.plan import GeneratorSettings, build_plan, describe_plan, load_profiles

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)
for noisy in ("py4j", "pyspark"):
    logging.getLogger(noisy).setLevel(logging.WARNING)

settings = GeneratorSettings(scale=SCALE, min_rows=MIN_ROWS, max_rows=MAX_ROWS, seed=SEED)
profiles, relationships = load_profiles(PROFILES)
plans = build_plan(profiles, relationships, settings)
print(describe_plan(plans, settings))

In [ ]:
# ============================== 5. GENERATE and write the tables =================================
from synthgen.generator.generate import generate
from synthgen.generator.write import write_tables

data = generate(plans, settings)
report = write_tables(spark, plans, data, TARGET_CATALOG, TARGET_SCHEMA, settings, table_format=TABLE_FORMAT,
                      declare_keys=DECLARE_KEYS, replace_existing=REPLACE_EXISTING)
del data
for table, rows in report.tables.items():
    print(f"{table}: {rows} rows")
for warning in report.warnings:
    print("warning:", warning)

In [ ]:
# ============================== 6. CHECK: row counts and links between tables ====================
from synthgen.generator.check import check_tables

for line in check_tables(spark, plans, TARGET_CATALOG, TARGET_SCHEMA):
    print(line)

In [ ]:
# ============================== 7. LOOK at the data (fake rows, safe to display) =================
for table in report.tables:
    print(table)
    display(spark.table(table).limit(20))